In [ ]:
con.close()

In [ ]:
import duckdb

con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=True)

# List all tables and views
df_objects = con.execute("""
    SELECT table_schema, table_name, table_type 
    FROM information_schema.tables 
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_type, table_name;
""").df()

print(df_objects)
# con.close()

In [ ]:
Purchases_Invoice = con.execute("SELECT * FROM Purchases_Invoice limit 20").df()
display(Purchases_Invoice)

In [ ]:
df.info()

In [ ]:
Sale_Invoice = con.execute("SELECT * FROM Sale_Invoice limit 20").df()
display(Sale_Invoice)

In [ ]:
Sale_Invoice.info()

In [ ]:
sql = """
SELECT "Customer Full Name", SUM("Earned Profit on Invoice") AS Total_Profit FROM "Sale_Invoice" WHERE "Invoice Year Number" = 2016 GROUP BY "Invoice Month Name"
"""
print(sql)
output = con.execute(sql).df()
display(output)

### Iceburg to SQLite

In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = r"C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

# staging_table_name = "staging.Integration.employee_Staging"
# wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    # print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)


In [ ]:
import sqlite3
from pyspark.sql.types import DateType, TimestampType, TimestampNTZType, DecimalType
from pyspark.sql.functions import col

SQLITE_DB_PATH = "WideWorldImporters.db"

namespaces = [row["namespace"] for row in spark.sql("SHOW NAMESPACES IN reporting").collect()]

for ns in namespaces:
    tables = spark.sql(f"SHOW TABLES IN reporting.{ns}").collect()
    
    for t in tables:
        table_name = t["tableName"]
        full_iceberg_name = f"reporting.{ns}.{table_name}"
        sqlite_table_name = f"{ns}_{table_name}"
        
        print(f"Exporting {full_iceberg_name} -> {sqlite_table_name}...")
        
        # Read table
        df = spark.read.table(full_iceberg_name)
        
        # Cast dates, timestamps, and decimals to prevent Windows OS and sqlite3 binding errors
        for field in df.schema.fields:
            # Fix 1: Handle pre-1970/out-of-range dates on Windows
            if isinstance(field.dataType, (DateType, TimestampType, TimestampNTZType)):
                df = df.withColumn(field.name, col(field.name).cast("string"))
            
            # Fix 2: Handle DecimalType for SQLite binding
            # Use "double" for standard math/numeric support in SQLite
            # Use "string" if absolute financial precision is required
            elif isinstance(field.dataType, DecimalType):
                df = df.withColumn(field.name, col(field.name).cast("double"))
        
        # Safely convert to Pandas and export to SQLite
        pdf = df.toPandas()
        
        with sqlite3.connect(SQLITE_DB_PATH) as conn:
            pdf.to_sql(sqlite_table_name, conn, if_exists="replace", index=False)

print("Export to SQLite completed successfully!")

sqlite3.close

In [ ]:
import os

db_filename = "WideWorldImporters.db"
absolute_path = os.path.abspath(db_filename)

print("SQLite DB Full Path:", absolute_path)

In [ ]:
spark.stop()

In [ ]:
import sqlite3

SQLITE_DB_PATH = r"C:\Users\progr\Downloads\WideWorldImporters.db"

with sqlite3.connect(SQLITE_DB_PATH) as conn:
    cursor = conn.cursor()
    
    # Query sqlite_master for all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    tables = cursor.fetchall()
    
    print(f"Total Tables Found: {len(tables)}\n" + "-"*35)
    for t in tables:
        table_name = t[0]
        # Get row count for each table
        cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
        row_count = cursor.fetchone()[0]
        print(f"• {table_name:<30} ({row_count:,} rows)")

In [ ]:
import sqlite3
import pandas as pd

SQLITE_DB_PATH = r"C:\Users\progr\Downloads\WideWorldImporters.db"

with sqlite3.connect(SQLITE_DB_PATH) as conn:
    # 1. Read table into a Pandas DataFrame
    df_orders = pd.read_sql_query("SELECT * FROM fact_order LIMIT 5", conn)
    print("--- First 5 Orders ---")
    print(df_orders.head())

    # 2. Perform a JOIN directly in SQL and load result into DataFrame
    join_query = """
    SELECT 
        o.OrderID,
        o.OrderDateKey,
        c.Customer,
        o.TotalIncludingTax
    FROM fact_order o
    LEFT JOIN dimension_customer c ON o.CustomerKey = c.CustomerKey
    LIMIT 10
    """
    df_order_customers = pd.read_sql_query(join_query, conn)
    
    print("\n--- Order & Customer Joined DataFrame ---")
    display(df_order_customers)

In [6]:
from pyspark.sql import SparkSession

# Stop any lingering background Spark contexts
try:
    spark.stop()
except:
    pass

SQLITE_JAR = "C:/data/spark/jars/sqlite-jdbc-3.36.0.3.jar"

spark = SparkSession.builder \
    .appName("SQLite Reader") \
    .config("spark.jars", SQLITE_JAR) \
    .getOrCreate()

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
from pyspark.sql import SparkSession

# Ensure you specify the path to your sqlite-jdbc jar
SQLITE_JAR = "C:/data/spark/jars/sqlite-jdbc-3.36.0.3.jar"  # Adjust path to your jar location

# 1. Initialize or get the existing SparkSession
spark = SparkSession.builder \
    .appName("SQLite Reader") \
    .config("spark.jars", SQLITE_JAR) \
    .getOrCreate()

# 2. Now run your SQLite JDBC code safely
SQLITE_DB_PATH = r"C:\Users\progr\Downloads\WideWorldImporters.db"
jdbc_url = f"jdbc:sqlite:{SQLITE_DB_PATH}"

df_fact_order = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "fact_order") \
    .option("driver", "org.sqlite.JDBC") \
    .load()

df_dim_customer = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "dimension_customer") \
    .option("driver", "org.sqlite.JDBC") \
    .load()

# Join DataFrames
df_joined = df_fact_order.join(
    df_dim_customer, 
    on="CustomerKey", 
    how="left"
).select(
    "OrderID", 
    "OrderDateKey", 
    "Customer", 
    "TotalIncludingTax"
)

df_joined.show(5, truncate=False)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
import sqlite3

SQLITE_DB_PATH = "WideWorldImporters.db"

# Define your SQL view creation query
CREATE_VIEW_SQL = """
CREATE VIEW IF NOT EXISTS v_customer_sales_summary AS
SELECT
"""

with sqlite3.connect(SQLITE_DB_PATH) as conn:
    cursor = conn.cursor()
    
    # 1. Create the view
    cursor.execute(CREATE_VIEW_SQL)
    print("View 'v_customer_sales_summary' created successfully!\n")
    
    # 2. Query data directly from the newly created view
    cursor.execute("SELECT * FROM v_customer_sales_summary LIMIT 5;")
    results = cursor.fetchall()
    
    # Display results
    print("Sample Output from View:")
    for row in results:
        print(row)